# A Practical Manual for the Engineering DSL

*Reflecting the current state of `circuit_dsl.py` and `calc_symbols.py`.*

This notebook is both a manual and a live tour: every feature is shown with a code cell you can execute. Run the setup cells once, then read top-to-bottom or jump around as needed.

## 1. What this is

The Engineering DSL is a Python source-code transformer that lets you write Jupyter cells the way an engineer writes calculations on paper. It is installed as an `ideas` import hook, so the transformations apply automatically to every cell after the hook is registered. There is no new language to learn — everything stays valid Python after the rewrite — but the input is allowed to use mathematical glyphs, implicit multiplication, postfix operators, subscript indices, vulgar fractions, and engineering unit symbols.

The implementation lives in two files. `circuit_dsl.py` performs the source rewrites and supplies the runtime helpers (`parallel`, `percent`, `fact`, `Range`, `plusminus`, `σ`, `Σ`, `sqrt`). `calc_symbols.py` defines the SI-prefixed engineering units (`kΩ`, `μF`, `MHz`, `mV`, …) and the constants `π` and `i`, building on top of `forallpeople`. A standalone module `i_mul_fys.py` exists as the original implicit-multiplication-only transformer; the canonical transformer is `circuit_dsl.py`, which subsumes it.

## 2. Setting up a notebook

Install the dependencies (only needed once per environment):

Register the hook and import the unit catalogue. The `forallpeople` environment must be initialised **before** `calc_symbols` is imported, because `calc_symbols` defines its prefixed units in terms of the SI base units (`A`, `Ohm`, `W`, `F`, `H`, `Hz`, `s`, `V`, `m`, `Celsius`) that `forallpeople` injects into the global namespace.

The hook only affects code that is executed *after* `add_hook()` returns, which is why the registration cell itself must remain plain Python.

In [1]:
from utils.Engineer import *

## 3. The transformation pipeline

When a cell is executed, the source string passes through the rewrites in `transform_source` in this fixed order:

1. Unicode normalisation (`normalize_source`)
2. Math assignment glyphs and bare-equals comparison (`rewrite_math_assignment`)
3. Parallel operator (`rewrite_parallel`)
4. Postfix percent (`rewrite_postfix_percent`)
5. Postfix permille (`rewrite_postfix_permille`)
6. Prefix square root (`rewrite_prefix_sqrt`)
7. Postfix factorial (`rewrite_postfix_factorial`)
8. Postfix superscripts ² and ³ (`rewrite_postfix_superscripts`)
9. Subscript indices (`rewrite_subscript_indices`)
10. Plus-minus operator (`rewrite_plusminus`)
11. Tokenisation, implicit multiplication insertion, untokenisation

The order matters. For example, the assignment rewrite happens before the parallel rewrite, so a line like `Z := R₁ ‖ R₂` first becomes `Z = R₁ ‖ R₂` (with `‖` already normalised to `||` in step 1), then the parallel rewrite folds it into `parallel(R[1], R[2])`.

You can inspect any transformation directly:

In [2]:
from utils.circuit_dsl import transform_source
print(transform_source("Z := R₁ ‖ R₂"))

Z = parallel(_idx(R, _S(1, _INF)), _idx(R, _S(2, _INF)))


## 4. Unicode normalisation

The first pass replaces a fixed set of Unicode glyphs with their ASCII equivalents so that the rest of the pipeline (and the Python tokenizer) can deal with familiar characters.

| Glyph | Becomes  | Notes                                   |
|-------|----------|-----------------------------------------|
| `←`   | `:=`     | Assignment arrow; later normalised to `=` |
| `°C`, `℃` | `degC` | Degree-Celsius literal                  |
| `·`, `⋅`, `×` | `*` | Multiplication dots and cross           |
| `−`   | `-`      | Unicode minus                           |
| `÷`   | `/`      | Division sign                           |
| `≠`   | `!=`     | Not-equal                               |
| `≤`   | `<=`     | Less-or-equal                           |
| `≥`   | `>=`     | Greater-or-equal                        |
| `‖`   | `\|\|`   | Parallel bars (math notation)           |
| `½ ⅓ ¼ ¾ ⅔ ⅕ ⅖ ⅗ ⅘ ⅙ ⅚ ⅛ ⅜ ⅝ ⅞` | `(1/2)` etc. | Vulgar fractions |
| `)(`, `) (`, `)  (` | `)*(` | Adjacent-parens implicit multiplication |

The `π` glyph is also given breathing room: when it is glued to adjacent identifiers or numbers (as in `2πr` or `π(r+1)`), spaces are inserted around it so that the tokenizer treats it as its own token. The line `π = pi` is left intact, so the constant can still be defined cleanly.

A consequence worth knowing: because `‖` is rewritten to `||` in step 1, the parallel rewrite in step 3 only needs to recognise the ASCII `||` form. You can write either `R1 || R2` or `R1 ‖ R2` and the result is identical.

In [3]:
# Degree-Celsius and the multiplication dots
22°C, 23℃

(22 °C, 23 °C)

In [4]:
# Unicode comparisons and Unicode minus
2 ≠ 3, 3 ≤ 8, 2 ≥ 9

(True, True, False)

## 5. Assignment versus comparison

Python uses `=` for assignment and `==` for comparison. Mathematical writing does the opposite — `=` is comparison, and assignment is written with an arrow or a colon-equals. The DSL recognises three assignment glyphs, all of which become `x = 5`:

```
x ← 5         # left-arrow
x ≔ 5         # colon-equals as a single Unicode glyph
x := 5        # ASCII colon-equals
```

Tuple assignment is supported on the left-hand side. Because assignment is unambiguously marked, a bare `=` at the top level of a line is treated as comparison and rewritten to `==`.

Two safety rails apply. First, the rewrite skips lines that begin with `def`, `class`, `import`, `from`, `for`, `with`, `except`, `lambda`, or `@`, so default-argument syntax, keyword arguments, and decorators are untouched. Second, an `=` inside parentheses, brackets, or braces is left alone, so calls like `f(x=1)` continue to work. Python's walrus `:=` is preserved when it is *not* at the start of a line — for example, `if (n := f()) > 0:` still parses as a walrus.

The decision rule: if the line begins with a target list followed by `≔`, `:=`, or `←`, that is an assignment. Otherwise, top-level bare `=` is comparison.

In [5]:
U1, P1 ← 6.0 V, 2.0 W
U1, P1

(6.0 V, 2.0 W)

In [6]:
I1 ← P1 / U1
I1

330 mA

In [7]:
R1 ← U1 / I1
R1

18. Ω

In [8]:
# Bare = at top level becomes ==
2 = 3

False

## 6. Implicit multiplication

After all the symbolic rewrites, the source is tokenised and a `*` is inserted between consecutive tokens that mathematicians would read as multiplication. The rules are deliberately conservative:

- A number followed by an identifier, another number, or `(` — `2n`, `2 3`, `2(x+1)` → `2*n`, `2*3`, `2*(x+1)`
- An identifier followed by an identifier or a number — `m n` → `m*n`
- A closing `)` followed by an identifier or a number — `(a+b)c` → `(a+b)*c`
- A closing `]` followed by an identifier, number, or `(`
- The specific case of `π(` — `π(r+1)` → `π*(r+1)`

Notably absent: an identifier followed by `(` is *not* rewritten, because in Python that is a function call. `f(x)` stays as `f(x)`. Only `π` is given an exception — an engineer is far more likely to write `π(r+1)` meaning `π·(r+1)` than to define a function called `π`.

A few consequences worth understanding. `2 3` becomes `2*3`, which evaluates to `6` — this can happen by accident, so be careful with stray whitespace in numeric literals. `7 ⅓` becomes `7*(1/3)` ≈ `2.33`, not the mixed-number reading 7⅓. `4 + 3i` works as expected because `i` is bound to `1j`.

In [9]:
r ← 1 m
r

1 m

In [10]:
# π as a bare factor — try the classic
2πr

6.28318530717959 m

In [11]:
# Stray whitespace in numbers
2 3

6

In [12]:
# The imaginary unit i is bound to 1j
4 + 3i

4 + 3*I

## 7. Postfix and prefix operators

### Percent and permille

```
25%        # percent(25)        → 0.25
x%         # percent(x)         → x / 100
(a+b)%     # percent((a+b))     → (a+b) / 100
5‰         # permille(5)        → 0.005
```

The percent rewrite is careful to leave the binary modulo operator alone: `x % y` stays as modulo. The trigger is a number, identifier, or parenthesised expression directly followed by `%`, *not* followed by another operand.

In [13]:
10 + 5%

10.05

In [14]:
3‰

0.003

### Factorial

```
5!       # fact(5)
x!       # fact(x)
(a+b)!   # fact((a+b))
2 != 3   # left alone — this is the not-equal operator
```

The factorial rewrite specifically excludes `!=`, so comparisons keep working. The `fact` helper accepts integers and floats that are exact integers (e.g. `5.0`); anything else raises `TypeError`.

In [15]:
pp('6! =',6!)

<IPython.core.display.Math object>

### Superscripts ² and ³

```
x²         # (x)**2
5²         # (5)**2
(a+b)²     # ((a+b))**2
```

In [16]:
n:= 3
r², 2¹², 3⁽ⁿ⁺¹⁾

(1 m², 4096, 81)

In [17]:
(8 + i)²

(8 + I)**2

### Square root

```
√144       # sqrt(144)
√x         # sqrt(x)
√(a+b)     # sqrt((a+b))
```

The `sqrt` helper is `x ** 0.5`, which means it works for both numbers and `forallpeople` quantities that support fractional exponents.

In [18]:
pp('√144=',√144)

<IPython.core.display.Math object>

### Subscript indices

A simple identifier directly followed by one or more subscript digits or sign markers is rewritten to a bracketed index. The full set of recognised subscript characters is `₀ ₁ ₂ ₃ ₄ ₅ ₆ ₇ ₈ ₉ ₊ ₋` plus the alphabetic subscripts `ₐ ₑ ₒ ₓ ₔ ₕ ₖ ₗ ₘ ₙ ₚ ₛ ₜ` (translated to plain letters inside the brackets).

```
x₁         # x[1]
x₁₀        # x[10]
x₋₁₂₃      # x[-123]
2x₃        # 2*x[3]   (combines with implicit multiplication)
```

In [19]:
x := [34, 35, 55, 100, 3, 4, 5, 6, 7, 89, 90] mV 
pp(x)
pp(x₁⇥x₃⇥x₁₀)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [20]:
2x₃

200 mV

In [21]:
4x₃ ≥ 400 mV

True

## 8. Parallel resistance and serial capacitance

The double-bar operator `||` (or its Unicode form `‖`) is rewritten to `parallel(a, b)`, which computes `1 / (1/a + 1/b)`. This is the parallel-resistance formula — and, because the algebra is identical, also the formula for series capacitance.

Operator precedence is whatever Python applies to the rewritten expression. `parallel(R1, R2) + 1.2` is a function call followed by an addition, which is fine. If you mean `parallel(R1, R2 + 1.2)`, write the parentheses explicitly: `R1 || (R2 + 1.2)`.

The rewrite recognises four shapes: `name||name`, `(expr)||name`, `name||(expr)`, and `(expr)||(expr)`. Anything more complex than a single parenthesised group on either side will not match — wrap the subexpression in a name first if the rewrite does not fire.

In [22]:
R2 := 12 kΩ
pp(R2)

<IPython.core.display.Math object>

In [23]:
R1 ‖ R2 + 1.2 Ω

19. Ω

In [24]:
C1, C2 ← 12 pF, 36 pF
C1 ‖ C2

9 pF

## 9. Tolerances and the `Range` type

The plus-minus operator `±` produces a `Range`:

```
1 ± 2                          # Range(-1, 3)
(2 ± 0.2) mV + (9 ± 0.5) V     # Range arithmetic preserved through unit math
```

`Range` represents an inclusive interval `[low, high]`. It is constructed by `plusminus(center, delta)` (which is `Range.from_pm`), or directly via `Range(low, high)`. If `low > high`, the constructor swaps them silently. A scalar `x` is automatically coerced into `Range(x, x)` when it appears in arithmetic with a `Range`.

The arithmetic implements interval arithmetic. Addition and subtraction add or subtract the endpoints in the appropriate order. Multiplication takes the minimum and maximum of the four corner products, which handles negative ranges correctly. Division does the same on the four corner ratios but raises `ZeroDivisionError` if the divisor range straddles zero, because the result would be unbounded. Negation produces `Range(-high, -low)`.

Two read-only properties are exposed: `r.center` (the midpoint) and `r.tol` (the half-width). The string representation uses the math symbol `‥`, e.g. `(1 ‥ 3)`.

A subtle point: interval arithmetic can be pessimistic. The expression `(x ± 1) - (x ± 1)` is `Range(-2, 2)`, not `Range(0, 0)`, because the two intervals are treated as independent unknowns. This is the standard behaviour of interval arithmetic and is the conservative answer.

In [25]:
(2 ± 0.2) mV + (9 ± 0.5) V

(8.50 V ‥ 9.50 V)

## 10. Engineering units and prefixes

`calc_symbols` defines a comprehensive set of prefixed SI units on top of `forallpeople`. The numeric prefixes are:

| Prefix | Value      | Prefix | Value         |
|--------|------------|--------|---------------|
| `prefix_p` | 1e-12 | `prefix_k` | 1e3       |
| `prefix_n` | 1e-9  | `prefix_M` | 1e6       |
| `prefix_μ` | 1e-6  | `prefix_G` | 1e9       |
| `prefix_m` | 1e-3  | `prefix_T` | 1e12      |
| `prefix_d` | 1e-2  |            |               |
| `prefix_c` | 1e-2  |            |               |

> **Note:** Both `prefix_d` and `prefix_c` are `1/100` in the current source, so `cm` and `dm` evaluate to the same length. If you need a true `prefix_d = 1/10`, this is the place to fix it.

The unit catalogue:

| Quantity     | Available units                                   |
|--------------|---------------------------------------------------|
| Current      | `pA`, `nA`, `μA`, `mA`, `A`                       |
| Resistance   | `mΩ`, `Ω`, `kΩ`, `MΩ`, `GΩ`                       |
| Power        | `nW`, `mW`, `W`                                   |
| Capacitance  | `pF`, `nF`, `μF`, `mF`, `F`                       |
| Inductance   | `pH`, `nH`, `μH`, `mH`, `H`                       |
| Frequency    | `Hz`, `kHz`, `MHz`, `GHz`, `THz`                  |
| Time         | `ps`, `ns`, `μs`, `s`                             |
| Voltage      | `pV`, `nV`, `μV`, `mV`, `V`                       |
| Length       | `mm`, `cm`, `m`                                   |
| Temperature  | `degC` (and `°C`/`℃` via Unicode normalisation)   |

Three additional fractional helpers are also exported: `ptm` (per-thousand, equal to `prefix_m`), `ptc` (per-hundred, equal to `prefix_d`), and `ppm` (parts-per-million, equal to `prefix_μ`).

In [26]:
pF

1 pF

In [27]:
4.47 μF

4.47 μF

In [28]:
nV

1 nV

In [29]:
L := 5 cm
L

5 cm

## 11. Constants and helper functions

Two constants are bound globally:

| Symbol | Value     |
|--------|-----------|
| `π`    | `math.pi` |
| `i`    | `1j`      |

Binding `i` to the imaginary unit means complex literals can be written as `4 + 3i`, which after implicit multiplication becomes `4 + 3*i = 4 + 3j`.

The runtime helpers exported from `circuit_dsl` are:

| Helper      | Behaviour                                                    |
|-------------|--------------------------------------------------------------|
| `parallel(x, y)` | `1 / (1/x + 1/y)` — parallel resistance / series capacitance |
| `percent(x)`     | `x / 100`                                                |
| `permille(x)`    | `x / 1000`                                               |
| `fact(x)`        | `math.factorial(x)`, also accepts integer-valued floats |
| `mod(x, y)`      | `x % y`                                                  |
| `plusminus(x, y)`| `Range.from_pm(x, y)`                                    |
| `σ(data, ddof=0)`| `numpy.std(data, ddof=ddof)` — population std by default |
| `Σ(data, axis=None)` | `numpy.sum(data, axis=axis)`                         |
| `sqrt(x)`        | `x ** 0.5`                                               |

`σ` defaults to the population standard deviation; pass `ddof=1` for the sample estimate.

In [30]:
σ([2, 4, 4, 4, 5, 5, 7, 9] mA) 

2 mA

In [31]:
Σ([1, 2, 3])

6

## 12. Worked examples

A handful of small computations that combine several features at once.

In [32]:
# Vulgar fraction with explicit multiplication
⅓ · (1 + 2) · (3 + 5) · 9 mA

72 mA

In [33]:
# Same expression with implicit multiplication on the fractions and parens
⅓(1 + 2)(3 + 5) · 9 mA

72 mA

In [34]:
# Mixed-number gotcha: this is 7 * (1/3), NOT 7 + 1/3
7⅓, 7+⅓ 

(2.333333333333333, 7.333333333333333)

In [35]:
# Top-level := assignment chain
a := 1 + 2 + 3
a

6

In [36]:
# Bare = becomes == at top level
1 + 2 + 3 = 1 · 2 · 3

True

A larger, real-world example: enumerating transmit-pulse patterns against a fixed sample rate. Note that inside the list literal, `←` and `:=` are *not* treated as assignment — they are only recognised at the start of a line. So the tuples parse as ordinary Python with implicit multiplication on each `(10 MHz, ...)` pair.

In [37]:
fs ← 60 MHz
txpatterns ← [
    (10 MHz,   "TXPAT0 10 MHz 4 Pulses"),
    ( 5 MHz,   "TXPAT1 5 MHz 3 Pulses"),
    ( 4 MHz,   "TXPAT2 4 MHz 4 Pulses"),
    ( 5 MHz,   "TXPAT3 5 MHz 1 Pulse"),
    ( 6 MHz,   "TXPAT4 6 MHz 1 Pulse"),
    ( 7.5 MHz, "TXPAT5 7.5 MHz 1 Pulse"),
    (10 MHz,   "TXPAT6 10 MHz 1 Pulse"),
    (60 MHz,   "TXPAT7 Test Pulse")
]
for f, t in txpatterns:
    print(f"{f:8.1f} {1/f:8.1f} {fs/f:8.0f} {t}")

    10.0 MHz    100.0 ns        6 TXPAT0 10 MHz 4 Pulses
     5.0 MHz    200.0 ns       12 TXPAT1 5 MHz 3 Pulses
     4.0 MHz    250.0 ns       15 TXPAT2 4 MHz 4 Pulses
     5.0 MHz    200.0 ns       12 TXPAT3 5 MHz 1 Pulse
     6.0 MHz    166.7 ns       10 TXPAT4 6 MHz 1 Pulse
     7.5 MHz    133.3 ns        8 TXPAT5 7.5 MHz 1 Pulse
    10.0 MHz    100.0 ns        6 TXPAT6 10 MHz 1 Pulse
    60.0 MHz     16.7 ns        1 TXPAT7 Test Pulse


And a regular Python `def` is left completely alone — the rewrite skips lines starting with `def`, `class`, etc.

In [38]:
def factors(x):
    print("The factors of", x, "are:")
    for 1 ≤ k ≤ x:
        if mod(x, k) = 0:
            pp(k)

factors(80)

The factors of 80 are:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 13. Limitations and gotchas

A handful of edges deserve explicit mention.

**Stray whitespace in numbers.** The implicit-multiplication rule that turns `2 3` into `2*3` will silently merge stray-whitespace number literals. If you mean the integer `23`, write `23`. If you mean the product, the explicit `*` is harmless and clearer.

**Parallel rewrite is non-recursive.** The rewrite recognises a *single* parenthesised group on each side. `(R1 + R2) || (R3 + R4)` works; `((R1 + R2) + R5) || R6` does not because the outermost group spans nested parentheses and the regex used is non-recursive. The workaround is to bind the subexpression to a name first.

**`Range` does not implement comparisons.** Hashing, `bool()`, and ordering operators are not currently defined. Comparisons against scalars therefore raise `TypeError`. If you need to test whether a range contains a value, write `r.low <= x <= r.high` explicitly.

**`prefix_d == prefix_c`.** Both are `1/100` in `calc_symbols.py`. That is almost certainly a typo for the deci prefix; if you rely on `dm` being `0.1 m`, fix the value.

**Mixed numbers are not supported.** `7 ⅓` becomes `7 * (1/3)` ≈ `2.33`, not `7 + 1/3` ≈ `7.33`. Write `7 + ⅓` if you want the mixed-number value.

**Subscripts are zero-based.** `x₁` is `x[1]`, the *second* element. If your engineering notation expects one-based subscripts, you need to either shift the numbering or wrap the access in a helper.

**Higher superscripts are not rewritten.** Only `²` and `³`. For `⁴` and beyond, use `**4`.

**The hook is per-process.** If you restart the Jupyter kernel, you must re-run the registration cell before any other DSL code.

## 14. Extending the DSL

The pipeline in `transform_source` is a plain sequence of string transformations. Adding a new postfix or prefix glyph is mostly a matter of writing one more `re.sub`-based rewrite and inserting it at the appropriate point in `transform_source`. The conventions used by the existing rewrites are worth copying.

Each rewrite is wrapped in a `while previous != source: previous = source; source = re.sub(...)` loop, so that nested occurrences are handled by repeated application rather than by writing a recursive pattern. Each rewrite handles four shapes — number, identifier, single-parenthesised group on the left, single-parenthesised group on the right — and explicitly excludes ambiguous trailing characters with a lookahead (the way the factorial rewrite excludes `!=` with `(?!\s*=)`).

For new glyphs that should normalise to ASCII before any structural rewriting, add the entry to the `replacements` dict in `normalize_source`. For glyphs that introduce a new function call, define the helper at module level in `circuit_dsl.py` and add it to the `from utils.circuit_dsl import *` pattern by giving it a public name (no leading underscore).

When testing a new transformation, the simplest harness is to call `transform_source(...)` directly with sample input and inspect the returned string. Because each pass is pure-string-to-pure-string, every step is independently testable.

In [39]:
# Try it out
from utils.circuit_dsl import transform_source
sample ← '''
R₁, R₂ ← 1 kΩ, 2.2 kΩ
Z ← R₁ ‖ R₂
V ← (5 ± 0.1) V
I ← V / Z
'''
print(transform_source(sample))


_idx(R, _S(1, _INF)), _idx(R, _S(2, _INF)) =( _S(1, _INF)* _wu(kΩ, 'kΩ')),( _S(2.2, 2)* _wu(kΩ, 'kΩ'))
Z = parallel(_idx(R, _S(1, _INF)), _idx(R, _S(2, _INF))
V) =( (plusminus(_S(5, _INF), _S(0.1, 1)))* _wu(V, 'V'))
I = V / Z



---

*End of manual.*